In [1]:
pip install PyPDF2

Note: you may need to restart the kernel to use updated packages.


In [13]:
import PyPDF2

def extract_text(pdf_path):
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

text = extract_text(r"C:\Users\sbm20\Downloads\Kaggle\jet_engine_rr.pdf")
print(text[:1000])  # preview




ISBN 0 902121 2 35


© Rolls-Royce plc 1986
Fifth editionReprinted 1996 with revisions.All rights reserved. No part of this publication may be
reproduced or transmitted in any form or by any means includingphotocopying and recording or storing in a retrieval system ofany nature without the written permission of the copyright owner.Application for such permission should be addressed to:
The Technical Publications Department
Rolls-Royce plcDerbyEngland
Colour reproduction by 
GH Graphics Ltd.
Printed in Great Britain by 
Renault Printing Co Ltd Birmingham England B44 8BS
For Rolls-Royce plc 
Derby England
ISBN 0902121 235
AcknowledgementsThe following illustrations appear by kind permission of the
companies listed.
Rolls-Royce/Snecma Olympus page 11 
Rolls-Royce Turbomeca Ltd. Adour Mk102 page 45
AdourMk151 page 199 
RTM322 Turboshaft page 243
Boeing Commercial Airplane Company page 144 Turbo-Union Ltd. RB199 page 169 IAE International Aero Engines AG V2500 page 251
Contents
1 Basic me

In [15]:
#Chunk the text
def chunk_text(text, chunk_size=800):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

chunks = chunk_text(text)
print(len(chunks))

530


In [19]:
#Create embeddings
from openai import OpenAI
client = OpenAI(api_key="your_key")

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

chunk_embeddings = [get_embedding(chunk) for chunk in chunks]

In [31]:
#print(chunk_embeddings[0])

data = list(zip(chunks, chunk_embeddings))

In [37]:
#Retrieve relevant chunks
#We find most similar chunks using cosine similarity.

import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search(query, top_k=3):
    query_emb = get_embedding(query)
    
    scores = []
    for chunk, emb in data:
        score = cosine_similarity(query_emb, emb)
        scores.append((chunk, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    
    return [chunk for chunk, _ in scores[:top_k]]

In [35]:
def ask_pdf(question):
    relevant_chunks = search(question)

    context = "\n\n".join(relevant_chunks)

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question: {question}

Answer in simple terms.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [47]:
print(ask_pdf("How many topics are here?"))
print(ask_pdf("WHat is the first topic? And tell me in breif about it"))

There are 25 topics listed in the context.
The first topic is "Basic mechanics." This section introduces the fundamental principles of how gas turbine engines work. It explains the basic concepts of jet propulsion, including how air is used as a working fluid to produce thrust. It covers the initial development and understanding of gas turbine engines, highlighting that before the 1950s, this technology was not well known in aircraft propulsion.
